# JenTravel — Agent-Powered Travel Assistant
This notebook runs a function-calling chatbot that represents a fictional travel business. It can answer questions, collect leads, and record feedback.


In [10]:
!pip install -r requirements.txt

In [11]:
# ------ Required Imports ------#

import os, json, re
from openai import OpenAI
import gradio as gr
from pypdf import PdfReader

In [12]:
# ------ Load environment varibales ------#
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
print("API Key loaded:", os.getenv("OPENAI_API_KEY"))

In [14]:
BASE = "/content/"
DATA_DIR = os.path.join(BASE, "data")
os.makedirs(DATA_DIR, exist_ok=True)

SUMMARY_PATH = os.path.join(BASE, "summary.txt")
PDF_PATH = os.path.join(BASE, "about_business.pdf")

# ---- Load sources ----
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    BUSINESS_SUMMARY = f.read()

def extract_pdf_text(path: str) -> str:
    try:
        reader = PdfReader(path)
        pages = [p.extract_text() or "" for p in reader.pages]
        return "\n".join(pages).strip()
    except Exception as e:
        return f"(PDF text extraction failed: {e})"

ABOUT_PDF_TEXT = extract_pdf_text(PDF_PATH)

# ---- System prompt ----
SYSTEM_PROMPT = """You are Journeys by Jenny’s friendly travel assistant for customers in Lebanon.
Stay in character as a helpful, concise travel concierge.

Use ONLY the content from:
• business_summary.txt
• about_business.pdf (extracted text)
to answer questions about destinations, packages, prices, and visa notes.

If something is NOT in those materials, do NOT guess.
Instead:
1) Call the tool "record_feedback" with the user's exact question.
2) Tell the user you’ve noted their request for follow-up.

Always encourage interested users to share their name and email, and
use the tool "record_customer_interest" when they do.

Never invent policies or guarantees beyond the provided context.
"""

# ---- Tools (file logging) ----
LEADS_PATH = os.path.join(DATA_DIR, "leads.jsonl")
FEEDBACK_PATH = os.path.join(DATA_DIR, "feedback.jsonl")

def record_customer_interest(email: str, name: str, message: str):
    payload = {"email": email, "name": name, "message": message}
    with open(LEADS_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")
    print("[lead]", payload)
    return "Thanks! We’ve saved your details — a travel advisor will follow up shortly."

def record_feedback(question: str):
    payload = {"question": question}
    with open(FEEDBACK_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")
    print("[feedback]", payload)
    return "I’ve noted your question for our team — thanks!"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "record_customer_interest",
            "description": "Save a potential customer's contact info and interest message.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string"},
                    "name": {"type": "string"},
                    "message": {"type": "string"}
                },
                "required": ["email", "name", "message"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_feedback",
            "description": "Log a question we can’t answer from current materials.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"}
                },
                "required": ["question"]
            }
        }
    }
]

# ---- OpenAI client ----
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

# ---- Helper: Build prompt with context ----
def build_messages(user_text: str, history_pairs: list):
    # history_pairs is [[user, assistant], ...] from Gradio
    context = (
        "business_summary.txt:\n" + BUSINESS_SUMMARY.strip() +
        "\n\nabout_business.pdf (text):\n" + ABOUT_PDF_TEXT.strip()
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",
         "content": f"Context:\n{context}\n\nNow answer the user (Lebanon-based customer): {user_text}"}
    ]
    # Attach prior turns for continuity
    for pair in history_pairs:
        if isinstance(pair, (list, tuple)) and len(pair) == 2:
            u, a = pair
            if u:
                messages.append({"role": "user", "content": u})
            if a:
                messages.append({"role": "assistant", "content": a})
    return messages

# ---- Tool dispatcher ----
def handle_tool_call(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments or "{}")
    if name == "record_customer_interest":
        return record_customer_interest(**args)
    if name == "record_feedback":
        return record_feedback(**args)
    return "Unknown tool."

# ---- Main chat logic ----
def respond(message, history):
    messages = build_messages(message, history)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto"
    )
    msg = resp.choices[0].message

    # If the model decided to call a tool
    if msg.tool_calls:
        call = msg.tool_calls[0]
        tool_result = handle_tool_call(call)

        follow = client.chat.completions.create(
            model=MODEL,
            messages=messages + [
                {"role": "assistant", "tool_calls": msg.tool_calls, "content": None},
                {"role": "tool", "tool_call_id": call.id, "name": call.function.name, "content": tool_result},
            ]
        )
        return follow.choices[0].message.content

    # Otherwise just return the text
    return msg.content

# ---- Light email validation (nudges tool usage) ----
EMAIL_HINT = re.compile(r"[^@\s]+@[^@\s]+\.[^@\s]+")

def chat_fn(user_input, history):
    # Nudge lead-capture when user drops email/name
    if EMAIL_HINT.search(user_input) and "name" in user_input.lower():
        # The model itself will likely call the tool, but this
        # encourages the behavior via content
        user_input += (
            "\n\n(If helpful, please call record_customer_interest with my email, name, and message.)"
        )
    return respond(user_input, history)






In [15]:
# ---- JenTravel: Teal Frontend (works with latest Gradio) ----
import gradio as gr

# 1) Theme: teal primary + cyan secondary, then force key tokens
theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="cyan",
    neutral_hue="slate",
    text_size="md",
    radius_size="md",
    spacing_size="lg",
).set(
    # Buttons
    button_primary_background_fill="linear-gradient(90deg, #0f766e, #14b8a6)",  # teal -> cyan
    button_primary_text_color="white",
    button_secondary_background_fill_hover="rgba(20, 184, 166, 0.12)",
    # Inputs
    input_background_fill="#ffffff",
    input_border_color="#cceae6",
    # Links
    link_text_color="#0f766e",
    link_text_color_active="#0d9488",
    # Blocks
    block_title_text_color="#0f766e",
    block_label_text_color="#0d9488",
)

# 2) Header + CSS to make the teal really show
HEADER_HTML = """
<style>
  /* Teal gradient header */
  .jentravel-hero {
    background: linear-gradient(135deg, #0f766e 0%, #14b8a6 100%);
    color: #fff; padding: 22px; border-radius: 14px; margin-bottom: 12px;
    box-shadow: 0 10px 30px rgba(15,118,110,.25);
  }
  .jentravel-hero h1 { margin: 0 0 6px 0; font-size: 1.8rem; }
  .jentravel-hero p { margin: 0; opacity: .95; }

  /* Chat bubble polish */
  .gr-chatbot { background: #f8fafc; } /* subtle neutral bg */
  /* Assistant bubble */
  .gr-chat-message.bot .bubble {
    background: linear-gradient(135deg, rgba(20, 184, 166, .12), rgba(20, 184, 166, .06));
    border: 1px solid rgba(20,184,166,.25);
  }
  /* User bubble */
  .gr-chat-message.user .bubble {
    background: #ffffff;
    border: 1px solid #e2e8f0;
  }

  /* Textbox focus ring in teal */
  textarea:focus, input:focus {
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(20, 184, 166, .25) !important;
    border-color: #14b8a6 !important;
  }
</style>

<div class="jentravel-hero">
  <h1>✈️ JenTravel</h1>
  <p>Your personal travel concierge from Lebanon — tailor-made trips, smart prices, and visa guidance.</p>
</div>
"""

DESCRIPTION_MD = (

    "**Ready to plan your next trip?** Let’s make it unforgettable! 🌴"
)

# Build the app as Blocks so we can add the header HTML above the ChatInterface
with gr.Blocks(theme=theme) as demo:
    gr.HTML(HEADER_HTML)
    gr.ChatInterface(
        fn=chat_fn,
        title="🌍 JenTravel — Your Personal Travel Concierge",
        description=DESCRIPTION_MD,
        examples=[
            ["What packages do you have for Turkey?"],
            ["Estimate the price for 2 people, 5 days in Sharm El Sheikh."],
            ["Do I need a visa for Europe?"],
            ["I’d like to book a honeymoon trip!"],
        ],
        chatbot=gr.Chatbot(
            show_label=False,
            show_copy_button=True,
            height=600,
            type="messages",  # required in new Gradio
            avatar_images=(
                "https://cdn-icons-png.flaticon.com/512/744/744502.png",  # user
                "https://cdn-icons-png.flaticon.com/512/888/888879.png",  # bot
            ),
        ),
        textbox=gr.Textbox(
            placeholder="Type your message here... 💬",
            label="Chat with JenTravel Assistant",
        ),
    )

# in Colab:
demo.launch(share=True, debug=True)


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'messages', will be used.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5be04876d7da3421bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5be04876d7da3421bf.gradio.live
